In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import torch
import pandas as pd

print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
from pathlib import Path

PROJECT = Path("/kaggle/working/vlm_research")
DATA_DIR = PROJECT / "data"
RESULTS_DIR = PROJECT / "results"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"

for folder in [DATA_DIR, RESULTS_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Data:", DATA_DIR)
print("Results:", RESULTS_DIR)
print("Checkpoints:", CHECKPOINT_DIR)

In [ ]:
%pip install -q -U accelerate qwen-vl-utils pycocotools scikit-learn matplotlib pillow tqdm
%pip install -q git+https://github.com/huggingface/transformers

In [ ]:
import torch
import transformers
import pandas as pd

print("Transformers:", transformers.__version__)
print("Pandas:", pd.__version__)
print("GPU:", torch.cuda.is_available())

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

print("Qwen model imports successful")

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("instances_val2017.json"):
    print("Annotation file:", p)

for p in Path("/kaggle/input").glob("*"):
    
    print("Dataset folder:", p)

In [ ]:
from pathlib import Path

COCO_ROOT = Path(
    "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
)

ANN_FILE = COCO_ROOT / "annotations" / "instances_val2017.json"
IMG_DIR = COCO_ROOT / "val2017"

print("Annotations exist:", ANN_FILE.exists())
print("Image folder exists:", IMG_DIR.exists())

if IMG_DIR.exists():
    images = list(IMG_DIR.glob("*.jpg"))
    print("Number of validation images:", len(images))
    if images:
        print("Example image:", images[0].name)

if ANN_FILE.exists():
    print("Annotation file size (MB):",
          round(ANN_FILE.stat().st_size / (1024 * 1024), 2))

In [ ]:
from pathlib import Path

PROJECT = Path("/kaggle/working/vlm_research")
DATA_DIR = PROJECT / "data"
RESULTS_DIR = PROJECT / "results"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"

for folder in [DATA_DIR, RESULTS_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders ready:", PROJECT)

In [ ]:
import json
import random
import pandas as pd
from collections import defaultdict

SEED = 42
N_IMAGES = 100
random.seed(SEED)

with open(ANN_FILE, "r") as f:
    coco = json.load(f)

categories = {
    item["id"]: item["name"]
    for item in coco["categories"]
}

annotations_by_image = defaultdict(set)

for ann in coco["annotations"]:
    if ann.get("iscrowd", 0) == 0:
        annotations_by_image[ann["image_id"]].add(
            ann["category_id"]
        )

image_info = {
    item["id"]: item
    for item in coco["images"]
}

candidates = [
    image_id
    for image_id, present in annotations_by_image.items()
    if len(present) >= 2
    and image_id in image_info
]

random.shuffle(candidates)

rows = []

for image_id in candidates:
    present = annotations_by_image[image_id]
    absent = set(categories) - present

    if not absent:
        continue

    im = image_info[image_id]
    image_path = IMG_DIR / im["file_name"]

    if not image_path.exists():
        continue

    positive_id = random.choice(sorted(present))
    negative_id = random.choice(sorted(absent))

    for category_id, label in [
        (positive_id, "yes"),
        (negative_id, "no")
    ]:
        rows.append({
            "image_id": image_id,
            "file_name": im["file_name"],
            "image_path": str(image_path),
            "object": categories[category_id],
            "question": (
                f"Is there a {categories[category_id]} in the image?"
            ),
            "label": label
        })

    if len({r["image_id"] for r in rows}) >= N_IMAGES:
        break

manifest = pd.DataFrame(rows)

manifest_path = DATA_DIR / "experiment_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Unique images:", manifest["image_id"].nunique())
print("Questions:", len(manifest))
print("Positive labels:", (manifest["label"] == "yes").sum())
print("Negative labels:", (manifest["label"] == "no").sum())
print("Manifest saved:", manifest_path)

display(manifest.head(6))

In [ ]:
from IPython.display import display, FileLink

print("Manifest exists:", manifest_path.exists())
print("Manifest size:", manifest_path.stat().st_size, "bytes")

display(FileLink(str(manifest_path)))

In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor
)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

assert torch.cuda.is_available(), "GPU is not available!"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated (GB):",
    round(torch.cuda.memory_allocated(0) / 1e9, 2)
)

In [ ]:
from PIL import Image, ImageFilter
from qwen_vl_utils import process_vision_info

def predict(image_path, question, blurred=False):
    with Image.open(image_path) as im:
        image = im.convert("RGB")

    if blurred:
        image = image.filter(
            ImageFilter.GaussianBlur(radius=3)
        )

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": (
                    question
                    + " Answer Yes or No based only on the visible image. "
                    + "If uncertain, say Uncertain."
                )
            }
        ]
    }]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False
        )

    generated = output[:, inputs.input_ids.shape[1]:]

    answer = processor.batch_decode(
        generated,
        skip_special_tokens=True
    )[0].strip()

    lower = answer.lower().strip(" .,!")

    if lower == "yes" or lower.startswith(("yes,", "yes ")):
        prediction = "yes"
    elif lower == "no" or lower.startswith(("no,", "no ")):
        prediction = "no"
    else:
        prediction = "uncertain"

    return answer, prediction


sample = manifest.iloc[0]

print("Question:", sample["question"])
print("Ground-truth label:", sample["label"])
print("Original:", predict(sample["image_path"], sample["question"]))
print("Blurred:", predict(
    sample["image_path"], sample["question"], blurred=True
))

In [ ]:
import pandas as pd
import time
from tqdm.auto import tqdm

RESULTS_PATH = RESULTS_DIR / "predictions.csv"
CHECKPOINT_PATH = CHECKPOINT_DIR / "predictions_checkpoint.csv"

if CHECKPOINT_PATH.exists():
    results = pd.read_csv(CHECKPOINT_PATH).to_dict("records")
elif RESULTS_PATH.exists():
    results = pd.read_csv(RESULTS_PATH).to_dict("records")
else:
    results = []

# Resume only successful predictions.
completed = {
    (int(r["image_id"]), r["object"], r["condition"])
    for r in results
    if r.get("prediction") in ("yes", "no", "uncertain")
    and not r.get("error", "")
}

start = time.time()
count = 0

for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
    for condition in ["original", "blurred"]:
        key = (int(row["image_id"]), row["object"], condition)

        if key in completed:
            continue

        try:
            raw, pred = predict(
                row["image_path"],
                row["question"],
                blurred=(condition == "blurred")
            )
            error = ""
        except Exception as e:
            raw, pred, error = "", "error", str(e)

        results.append({
            **row.to_dict(),
            "condition": condition,
            "raw_answer": raw,
            "prediction": pred,
            "error": error
        })

        if not error:
            completed.add(key)

        count += 1

        if count % 10 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)
            print(f"Checkpoint saved. Successful predictions: {len(completed)}")

    pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_PATH, index=False)

print("Inference finished.")
print("Rows:", len(results_df))
print("Successful predictions:", len(completed))
print("Elapsed minutes:", round((time.time() - start) / 60, 1))
print("Final results:", RESULTS_PATH)

In [ ]:
print(results_df["condition"].value_counts())
print("\nPrediction counts:")
print(results_df.groupby(["condition", "prediction"]).size())
print("\nErrors:")
print(results_df[results_df["error"].fillna("") != ""][
    ["image_id", "object", "condition", "error"]
].head(10))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_condition(group):
    valid = group[
        group["prediction"].isin(["yes", "no"])
        & group["label"].isin(["yes", "no"])
        & group["error"].fillna("").eq("")
    ]

    y_true = valid["label"]
    y_pred = valid["prediction"]

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=["no", "yes"]
    ).ravel()

    return {
        "total": len(group),
        "evaluated_yes_no": len(valid),
        "uncertain_or_error": len(group) - len(valid),
        "uncertainty_rate": (
            (len(group) - len(valid)) / len(group)
            if len(group) else np.nan
        ),
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "precision": precision_score(
            y_true, y_pred, pos_label="yes", zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, pos_label="yes", zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, pos_label="yes", zero_division=0
        ),
        "FPR": fp / (fp + tn) if (fp + tn) else np.nan
    }

summary = pd.DataFrame({
    condition: evaluate_condition(group)
    for condition, group in results_df.groupby("condition")
}).T

display(summary.round(4))

summary.to_csv(RESULTS_DIR / "metrics_summary.csv")
print("Saved metrics:", RESULTS_DIR / "metrics_summary.csv")

In [ ]:
import shutil
from pathlib import Path

archive_base = "/kaggle/working/vlm_research_results"

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=PROJECT
)

print("Results archive:", archive_path)

In [ ]:
import pandas as pd

# Match original and blurred predictions for the same question
paired = results_df.pivot_table(
    index=["image_id", "object", "question", "label"],
    columns="condition",
    values="prediction",
    aggfunc="first"
).reset_index()

def is_correct(pred, label):
    return pred == label

paired["original_correct"] = paired.apply(
    lambda r: is_correct(r["original"], r["label"])
    if r["original"] in ["yes", "no"] else False,
    axis=1
)

paired["blurred_correct"] = paired.apply(
    lambda r: is_correct(r["blurred"], r["label"])
    if r["blurred"] in ["yes", "no"] else False,
    axis=1
)

print("Original correct, blurred incorrect:",
      ((paired["original_correct"]) &
       (~paired["blurred_correct"])).sum())

print("Original incorrect, blurred correct:",
      ((~paired["original_correct"]) &
       (paired["blurred_correct"])).sum())

print("Both correct:",
      (paired["original_correct"] &
       paired["blurred_correct"]).sum())

print("Both incorrect or uncertain:",
      ((~paired["original_correct"]) &
       (~paired["blurred_correct"])).sum())

print("\nPrediction transitions:")
display(pd.crosstab(
    paired["original"],
    paired["blurred"],
    rownames=["Original prediction"],
    colnames=["Blurred prediction"],
    margins=True
))

paired.to_csv(RESULTS_DIR / "paired_analysis.csv", index=False)
print("Saved paired analysis.")

In [ ]:
print(
    results_df.groupby(["condition", "label", "prediction"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
from pathlib import Path
import shutil

BACKUP_DIR = Path("/kaggle/working/vlm_pilot_backup")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

files_to_backup = {
    "manifest.csv": DATA_DIR / "experiment_manifest.csv",
    "predictions.csv": RESULTS_DIR / "predictions.csv",
    "metrics_summary.csv": RESULTS_DIR / "metrics_summary.csv",
}

for name, source in files_to_backup.items():
    if source.exists():
        shutil.copy2(source, BACKUP_DIR / name)
        print("Backed up:", name)
    else:
        print("Not found:", source)

print("\nBackup folder:", BACKUP_DIR)

In [ ]:
import json
import random
import pandas as pd
from collections import defaultdict
from pathlib import Path

SEED = 42
random.seed(SEED)

# 1. Load COCO annotations
with open(ANN_FILE, "r") as f:
    coco = json.load(f)

# 2. Map category ID to category name
categories = {
    c["id"]: c["name"]
    for c in coco["categories"]
}

# 3. Collect annotated object categories for every image
ann_by_image = defaultdict(set)

for ann in coco["annotations"]:
    if ann.get("iscrowd", 0) == 0:
        ann_by_image[ann["image_id"]].add(
            ann["category_id"]
        )

# 4. Create one positive and one annotation-derived
#    negative question per eligible image
rows = []

for im in sorted(coco["images"], key=lambda x: x["id"]):
    image_id = im["id"]
    image_path = IMG_DIR / im["file_name"]

    if not image_path.exists():
        continue

    present = ann_by_image[image_id]
    annotation_absent = set(categories) - present

    if not present or not annotation_absent:
        continue

    positive_id = random.choice(sorted(present))
    negative_id = random.choice(sorted(annotation_absent))

    for category_id, label in [
        (positive_id, "yes"),
        (negative_id, "no")
    ]:
        rows.append({
            "image_id": image_id,
            "file_name": im["file_name"],
            "image_path": str(image_path),
            "object": categories[category_id],
            "question": (
                f"Is there a {categories[category_id]} "
                "in the image?"
            ),
            "label": label,
            "seed": SEED
        })

large_manifest = pd.DataFrame(rows)

LARGE_MANIFEST_PATH = (
    DATA_DIR / "large_experiment_manifest.csv"
)
large_manifest.to_csv(LARGE_MANIFEST_PATH, index=False)

print("Unique images:", large_manifest["image_id"].nunique())
print("Total questions:", len(large_manifest))
print("\nLabel counts:")
print(large_manifest["label"].value_counts())
print("\nSaved:", LARGE_MANIFEST_PATH)

display(large_manifest.head(6))

In [ ]:
from pathlib import Path
import shutil

BACKUP_DIR = Path("/kaggle/working/vlm_research_backup")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    LARGE_MANIFEST_PATH,
    BACKUP_DIR / "large_experiment_manifest.csv"
)

backup_zip = shutil.make_archive(
    "/kaggle/working/vlm_manifest_backup",
    "zip",
    root_dir=BACKUP_DIR
)

print("Manifest backup:", backup_zip)

In [ ]:
if "timing_df" in globals():
    display(timing_df.head())
    print("Predictions:", len(timing_df))
    print("Errors:", timing_df["error"].fillna("").ne("").sum())
else:
    print("timing_df does not exist. The timing cell may not have completed.")

In [ ]:
print("Manifest exists:", "large_manifest" in globals())
print("Predict function exists:", "predict" in globals())
print("Model exists:", "model" in globals())
print("Processor exists:", "processor" in globals())

if "large_manifest" in globals():
    print("Manifest rows:", len(large_manifest))
    print("Unique images:", large_manifest["image_id"].nunique())

In [ ]:
import time
import pandas as pd
from tqdm.auto import tqdm

test_ids = large_manifest["image_id"].drop_duplicates().head(10).tolist()
timing_manifest = large_manifest[
    large_manifest["image_id"].isin(test_ids)
].copy()

timing_results = []
start_time = time.time()

for _, row in tqdm(
    timing_manifest.iterrows(),
    total=len(timing_manifest),
    desc="Timing test"
):
    for condition in ["original", "blurred"]:
        try:
            raw, pred = predict(
                row["image_path"],
                row["question"],
                blurred=(condition == "blurred")
            )
            error = ""
        except Exception as e:
            raw, pred = "", "error"
            error = repr(e)

        timing_results.append({
            **row.to_dict(),
            "condition": condition,
            "raw_answer": raw,
            "prediction": pred,
            "error": error
        })

timing_df = pd.DataFrame(timing_results)

timing_path = RESULTS_DIR / "timing_test_10_images.csv"
timing_df.to_csv(timing_path, index=False)

elapsed = time.time() - start_time

print("Images:", len(test_ids))
print("Questions:", len(timing_manifest))
print("Predictions:", len(timing_df))
print("Errors:", timing_df["error"].ne("").sum())
print("Elapsed minutes:", round(elapsed / 60, 2))
print("Average seconds per prediction:", round(elapsed / len(timing_df), 2))
print("Saved to:", timing_path)

In [ ]:

import time
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path

checkpoint_path = RESULTS_DIR / "large_predictions_checkpoint.csv"

# Load any existing checkpoint
if checkpoint_path.exists():
    old_df = pd.read_csv(checkpoint_path)
    records = {}

    for r in old_df.to_dict("records"):
        key = (str(int(r["image_id"])), str(r["object"]), str(r["condition"]))
        records[key] = r

    print("Existing saved predictions:", len(records))
else:
    records = {}
    print("Starting fresh.")

def make_key(image_id, obj, condition):
    return (str(int(image_id)), str(obj), str(condition))

# Successful predictions don't need to be repeated
completed = {
    key for key, r in records.items()
    if str(r.get("error", "")) in ("", "nan", "None")
    and str(r.get("prediction", "")) not in ("error", "nan", "")
}

pending = []

for _, row in large_manifest.iterrows():
    for condition in ["original", "blurred"]:
        key = make_key(row["image_id"], row["object"], condition)
        if key not in completed:
            pending.append((row, condition, key))

print("Expected total predictions:", len(large_manifest) * 2)
print("Already completed successfully:", len(completed))
print("Predictions remaining:", len(pending))

start = time.time()
since_save = 0

for row, condition, key in tqdm(pending, desc="Full experiment"):
    try:
        raw, pred = predict(
            row["image_path"],
            row["question"],
            blurred=(condition == "blurred")
        )
        error = ""
    except Exception as e:
        raw, pred = "", "error"
        error = repr(e)

    records[key] = {
        **row.to_dict(),
        "condition": condition,
        "raw_answer": raw,
        "prediction": pred,
        "error": error
    }

    since_save += 1

    if since_save >= 200:
        pd.DataFrame(records.values()).to_csv(
            checkpoint_path, index=False
        )
        since_save = 0

# Save final progress
final_df = pd.DataFrame(records.values())
final_df.to_csv(checkpoint_path, index=False)

print("\nExperiment pass finished!")
print("Saved rows:", len(final_df))
print("Successful predictions:",
      final_df["error"].fillna("").eq("").sum())
print("Failed predictions:",
      final_df["error"].fillna("").ne("").sum())
print("Elapsed minutes:", round((time.time() - start) / 60, 2))
print("Saved to:", checkpoint_path)


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

# 1. Find CSV automatically
csv_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "large_predictions_checkpoint.csv" in files:
        csv_path = os.path.join(root, "large_predictions_checkpoint.csv")
        break

if csv_path is None:
    raise FileNotFoundError("CSV not found. Attach your Kaggle dataset.")

# 2. Load data
df = pd.read_csv(csv_path)
print("CSV:", csv_path)
print("Shape:", df.shape)

# 3. Clean columns
for col in ["label", "prediction", "condition"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

# 4. Calculate metrics
results = []

for condition in ["original", "blurred"]:
    data = df[df["condition"] == condition]
    binary = data[data["prediction"].isin(["yes", "no"])]

    y_true = binary["label"]
    y_pred = binary["prediction"]

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=["no", "yes"]
    ).ravel()

    results.append({
        "Condition": condition,
        "Total": len(data),
        "Binary_answers": len(binary),
        "Uncertain": len(data) - len(binary),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label="yes", zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label="yes", zero_division=0),
        "F1": f1_score(y_true, y_pred, pos_label="yes", zero_division=0),
        "False_Positive_Rate": fp / (fp + tn) if fp + tn else 0,
        "Uncertainty_Rate": (len(data) - len(binary)) / len(data) if len(data) else 0,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn
    })

metrics = pd.DataFrame(results)

# 5. Display results
print("\n===== METRICS =====")
print(metrics.round(4).to_string(index=False))

# 6. Save metrics CSV
metrics.to_csv("/kaggle/working/custom_metrics.csv", index=False)

# 7. Plot metric comparison
cols = [
    "Accuracy", "Precision", "Recall",
    "F1", "False_Positive_Rate", "Uncertainty_Rate"
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flat, cols):
    ax.bar(metrics["Condition"], metrics[col])
    ax.set_title(col.replace("_", " "))
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)

plt

In [ ]:
import os

print(os.listdir("/kaggle/working"))
print("Graph exists:",
      os.path.exists("/kaggle/working/metrics_comparison.png"))

In [ ]:
plt.savefig("/kaggle/working/metrics_comparison.png", dpi=300)
plt.show()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
from IPython.display import display, Image

# 1. Find the uploaded CSV
csv_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "large_predictions_checkpoint.csv" in files:
        csv_path = os.path.join(root, "large_predictions_checkpoint.csv")
        break

if csv_path is None:
    raise FileNotFoundError(
        "CSV not found. Check that 'for metric calculation' is attached."
    )

df = pd.read_csv(csv_path)

for col in ["label", "prediction", "condition"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

print("Loaded CSV:", csv_path)
print("Rows:", len(df))

# 2. Calculate metrics
results = []

for condition in ["original", "blurred"]:
    data = df[df["condition"] == condition]
    binary = data[data["prediction"].isin(["yes", "no"])]

    if binary.empty:
        print("No binary predictions for:", condition)
        continue

    y_true = binary["label"]
    y_pred = binary["prediction"]

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=["no", "yes"]
    ).ravel()

    results.append({
        "Condition": condition,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label="yes", zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label="yes", zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, pos_label="yes", zero_division=0),
        "False Positive Rate": fp / (fp + tn) if fp + tn else 0,
        "Uncertainty Rate": (len(data) - len(binary)) / len(data) if len(data) else 0
    })

metrics = pd.DataFrame(results)

if metrics.empty:
    raise ValueError("No metrics calculated. Check the condition and prediction values.")

display(metrics.round(4))

# 3. Create and save the graph
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, metrics.columns[1:]):
    ax.bar(metrics["Condition"], metrics[col])
    ax.set_title(col)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Rate")
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Effect of Gaussian Blur on VLM Performance", fontsize=16)
fig.tight_layout()

output_path = "/kaggle/working/metrics_comparison.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# 4. Save the metrics table and verify files
metrics.to_csv("/kaggle/working/custom_metrics.csv", index=False)

print("\nGraph created:", os.path.exists(output_path))
print("Graph location:", output_path)
print("Metrics CSV:", "/kaggle/working/custom_metrics.csv")

# Display saved PNG directly
display(Image(filename=output_path))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)

    if depth <= 2:
        print(root)
        for d in dirs:
            print("   📁", d)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "pope" in f.lower():
            print(os.path.join(root, f))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.startswith("coco_pope_") and f.endswith(".json"):
            print(os.path.join(root, f))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/awsaf49/coco-2017-dataset"):
    if "val2014" in root.lower():
        print("Found:", root)

In [ ]:
import os
import json

base = "/kaggle/input/datasets/umamaheshkamparaju/for-pope"

for name in [
    "coco_pope_random.json",
    "coco_pope_popular.json",
    "coco_pope_adversarial.json"
]:
    path = os.path.join(base, name)

    with open(path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    print(f"\n{name}")
    print("Number of questions:", len(data))
    print("First question:", data[0])

In [ ]:
import os

target = "COCO_val2014_000000310196.jpg"
matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    if target in files:
        matches.append(os.path.join(root, target))

if matches:
    IMAGE_DIR = os.path.dirname(matches[0])
    print("SUCCESS! Image found:", matches[0])
    print("IMAGE_DIR =", IMAGE_DIR)
else:
    print("Target image not found in attached datasets.")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    if "val2014" in root.lower():
        print("FOUND:", root)
        print("Sample files:", files[:5])

In [ ]:
import os
import subprocess

url = "https://images.cocodataset.org/zips/val2014.zip"
zip_path = "/kaggle/working/val2014.zip"

result = subprocess.run([
    "wget", "--no-check-certificate", "-c",
    url, "-O", zip_path
])

if result.returncode == 0 and os.path.getsize(zip_path) > 1_000_000:
    print("Download completed:", zip_path)
    print("Size (GB):", round(os.path.getsize(zip_path) / 1e9, 2))
else:
    print("Download failed or incomplete. Return code:", result.returncode)

In [ ]:
import os
import zipfile

zip_path = "/kaggle/working/val2014.zip"
extract_dir = "/kaggle/working/coco2014"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

IMAGE_DIR = os.path.join(extract_dir, "val2014")

print("Image directory exists:", os.path.isdir(IMAGE_DIR))
print("Sample images:", os.listdir(IMAGE_DIR)[:5])

target = os.path.join(IMAGE_DIR, "COCO_val2014_000000310196.jpg")
print("POPE sample image exists:", os.path.isfile(target))

In [ ]:
import os
import json
import torch
import pandas as pd
from PIL import Image, ImageFilter
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

# Paths
IMAGE_DIR = "/kaggle/working/coco2014/val2014"
POPE_FILE = "/kaggle/input/datasets/umamaheshkamparaju/for-pope/coco_pope_random.json"

# Load 100 questions
with open(POPE_FILE, "r") as f:
    questions = [json.loads(line) for line in f if line.strip()][:100]

# Load model
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

def predict(image, question):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",
             "text": question + " Answer only yes or no."}
        ]
    }]

    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[image], return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=5, do_sample=False)

    new_tokens = output[:, inputs.input_ids.shape[1]:]
    answer = processor.batch_decode(
        new_tokens, skip_special_tokens=True
    )[0].strip().lower()

    if answer.startswith("yes"):
        return "yes"
    if answer.startswith("no"):
        return "no"
    return "uncertain"

results = []
missing = 0

for i, item in enumerate(questions, 1):
    image_path = os.path.join(IMAGE_DIR, item["image"])

    if not os.path.isfile(image_path):
        missing += 1
        continue

    with Image.open(image_path) as im:
        original = im.convert("RGB")
        blurred = original.filter(ImageFilter.GaussianBlur(radius=3))

        for condition, image in [
            ("original", original),
            ("blurred", blurred)
        ]:
            answer = predict(image, item["text"])
            results.append({
                "question_id": item["question_id"],
                "image": item["image"],
                "question": item["text"],
                "label": item["label"].lower(),
                "condition": condition,
                "prediction": answer
            })

    if i % 10 == 0:
        print(f"Processed {i}/{len(questions)} questions")

df = pd.DataFrame(results)
output_path = "/kaggle/working/pope_random_pilot.csv"
df.to_csv(output_path, index=False)

print("\nPilot complete!")
print("Questions requested:", len(questions))
print("Missing images:", missing)
print("Predictions:", len(df))
print("Saved:", output_path)
print(df.groupby("condition")["prediction"].value_counts())

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("/kaggle/working/pope_random_pilot.csv")

# Normalize labels and predictions
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["prediction"] = df["prediction"].astype(str).str.strip().str.lower()

for condition in ["original", "blurred"]:
    d = df[df["condition"] == condition]

    y_true = d["label"]
    y_pred = d["prediction"]

    print(f"\n--- {condition.upper()} ---")
    print("Questions:", len(d))
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(
        y_true, y_pred, pos_label="yes", zero_division=0
    ), 4))
    print("Recall:", round(recall_score(
        y_true, y_pred, pos_label="yes", zero_division=0
    ), 4))
    print("F1:", round(f1_score(
        y_true, y_pred, pos_label="yes", zero_division=0
    ), 4))
    print("Yes ratio:", round((y_pred == "yes").mean(), 4))

In [ ]:
import os, json, torch, pandas as pd
from PIL import Image, ImageFilter
from tqdm.auto import tqdm

# Paths
POPE_FILE = "/kaggle/input/datasets/umamaheshkamparaju/for-pope/coco_pope_random.json"
IMAGE_DIR = "/kaggle/working/coco2014/val2014"
OUT_FILE = "/kaggle/working/pope_random_full.csv"
CHECKPOINT = "/kaggle/working/pope_random_checkpoint.csv"

# Load all POPE questions (JSONL format)
with open(POPE_FILE, "r") as f:
    questions = [json.loads(line) for line in f if line.strip()]

print("Questions loaded:", len(questions))

# Reuse pilot results as the starting checkpoint, if available
if os.path.exists("/kaggle/working/pope_random_pilot.csv"):
    existing = pd.read_csv("/kaggle/working/pope_random_pilot.csv")
else:
    existing = pd.DataFrame()

if not existing.empty:
    existing.to_csv(CHECKPOINT, index=False)

done = set()
if not existing.empty:
    done = set(zip(existing["question_id"], existing["condition"]))

# Check that the model and processor are loaded
assert "model" in globals() and "processor" in globals(), (
    "Model/processor aren't loaded in this session. Load them before running."
)

def classify_answer(answer):
    answer = answer.strip().lower()
    if answer.startswith("yes"):
        return "yes"
    if answer.startswith("no"):
        return "no"
    return "uncertain"

def predict_one(image, question):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question + " Answer only yes or no."}
        ]
    }]

    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[image], padding=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=8, do_sample=False)

    generated = output[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(
        generated, skip_special_tokens=True
    )[0]
    return classify_answer(answer)

# Write a checkpoint every 50 questions
rows = existing.to_dict("records") if not existing.empty else []
processed_ids = {qid for qid, _ in done}

for item in tqdm(questions, desc="POPE random"):
    qid = item["question_id"]
    if qid in processed_ids:
        continue

    image_path = os.path.join(IMAGE_DIR, item["image"])
    if not os.path.exists(image_path):
        print("Missing image:", image_path)
        continue

    try:
        with Image.open(image_path) as im:
            original = im.convert("RGB")

        blurred = original.filter(ImageFilter.GaussianBlur(radius=3))

        for condition, image in [
            ("original", original),
            ("blurred", blurred)
        ]:
            if (qid, condition) in done:
                continue

            prediction = predict_one(image, item["text"])
            rows.append({
                "question_id": qid,
                "image": item["image"],
                "question": item["text"],
                "label": item["label"].strip().lower(),
                "condition": condition,
                "prediction": prediction
            })
            done.add((qid, condition))

        processed_ids.add(qid)

        if len(processed_ids) % 50 == 0:
            pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

    except Exception as e:
        print(f"Error on question {qid}: {e}")
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
        raise

pd.DataFrame(rows).to_csv(OUT_FILE, index=False)
pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nCompleted!")
print("Predictions:", len(rows))
print("Saved:", OUT_FILE)
print(pd.DataFrame(rows).groupby(
    ["condition", "prediction"]
).size())

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

df = pd.read_csv("/kaggle/working/pope_random_full.csv")

for condition in ["original", "blurred"]:
    d = df[df["condition"] == condition].copy()
    y_true = d["label"].str.lower().str.strip()
    y_pred = d["prediction"].str.lower().str.strip()

    # Evaluate binary predictions; report uncertain separately
    valid = y_pred.isin(["yes", "no"])
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=["no", "yes"]
    ).ravel()

    print(f"\n--- {condition.upper()} ---")
    print("Evaluated:", len(y_true))
    print("Uncertain:", len(d) - len(y_true))
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(
        y_true, y_pred, pos_label="yes", zero_division=0
    ), 4))
    print("Recall:", round(recall_score(
        y_true, y_pred, pos_label="yes", zero_division=0
    ), 4))
    print("F1:", round(f1_score(
        y_true, y_pred, pos_label="yes", zero_division=0
    ), 4))
    print("FPR:", round(fp / (fp + tn), 4) if fp + tn else 0)
    print(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")

In [ ]:
import os, json, time, torch, pandas as pd
from PIL import Image, ImageFilter
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

IMAGE_DIR = "/kaggle/working/coco2014/val2014"
BASE = "/kaggle/working"

SPLITS = {
    "popular": f"{BASE}/../input/datasets/umamaheshkamparaju/for-pope/coco_pope_popular.json",
    "adversarial": f"{BASE}/../input/datasets/umamaheshkamparaju/for-pope/coco_pope_adversarial.json"
}

# Correct the paths explicitly
SPLITS = {
    "popular": "/kaggle/input/datasets/umamaheshkamparaju/for-pope/coco_pope_popular.json",
    "adversarial": "/kaggle/input/datasets/umamaheshkamparaju/for-pope/coco_pope_adversarial.json"
}

assert "model" in globals() and "processor" in globals(), \
    "Load your Qwen model and processor first."

def classify_answer(answer):
    answer = answer.strip().lower()
    if answer.startswith("yes"):
        return "yes"
    if answer.startswith("no"):
        return "no"
    return "uncertain"

# Same inference format as your successful pilot
def predict_one(image, question):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",
             "text": question + " Answer only yes or no."}
        ]
    }]

    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[image],
        padding=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=8, do_sample=False
        )

    generated = output[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(
        generated, skip_special_tokens=True
    )[0]
    return classify_answer(answer)

def calculate_metrics(csv_path, split_name):
    df = pd.read_csv(csv_path)
    summary = []

    for condition in ["original", "blurred"]:
        d = df[df["condition"] == condition].copy()
        valid = d["prediction"].isin(["yes", "no"])
        uncertain = int((~valid).sum())
        d = d[valid]

        y_true = d["label"].astype(str).str.lower().str.strip()
        y_pred = d["prediction"].astype(str).str.lower().str.strip()

        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=["no", "yes"]
        ).ravel()

        row = {
            "split": split_name,
            "condition": condition,
            "questions": len(df[df["condition"] == condition]),
            "evaluated": len(d),
            "uncertain": uncertain,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(
                y_true, y_pred, pos_label="yes", zero_division=0
            ),
            "recall": recall_score(
                y_true, y_pred, pos_label="yes", zero_division=0
            ),
            "f1": f1_score(
                y_true, y_pred, pos_label="yes", zero_division=0
            ),
            "false_positive_rate": fp / (fp + tn) if fp + tn else 0,
            "TP": tp, "FP": fp, "TN": tn, "FN": fn
        }
        summary.append(row)

        print(f"\n{split_name.upper()} — {condition.upper()}")
        for k, v in row.items():
            if k not in ["split", "condition"]:
                print(f"{k}: {v:.4f}" if isinstance(v, float)
                      else f"{k}: {v}")

    return summary

all_summaries = []
start_all = time.time()

for split_name, pope_path in SPLITS.items():
    output_file = f"{BASE}/pope_{split_name}_full.csv"
    checkpoint_file = f"{BASE}/pope_{split_name}_checkpoint.csv"

    with open(pope_path, "r") as f:
        questions = [json.loads(line) for line in f if line.strip()]

    print(f"\n{'='*50}")
    print(f"Starting {split_name}: {len(questions)} questions")
    print(f"{'='*50}")

    # Resume from checkpoint if available
    if os.path.exists(checkpoint_file):
        rows = pd.read_csv(checkpoint_file).to_dict("records")
        print("Resuming checkpoint with", len(rows), "predictions")
    else:
        rows = []

    done = {
        (int(r["question_id"]), r["condition"])
        for r in rows
    }

    for item in tqdm(questions, desc=f"POPE {split_name}"):
        qid = int(item["question_id"])
        image_name = item["image"]
        image_path = os.path.join(IMAGE_DIR, image_name)

        needed = [
            c for c in ["original", "blurred"]
            if (qid, c) not in done
        ]
        if not needed:
            continue

        if not os.path.exists(image_path):
            print("Missing image:", image_name)
            continue

        try:
            with Image.open(image_path) as im:
                original = im.convert("RGB")

            blurred = original.filter(ImageFilter.GaussianBlur(radius=3))

            for condition in needed:
                image = original if condition == "original" else blurred
                pred = predict_one(image, item["text"])

                rows.append({
                    "question_id": qid,
                    "image": image_name,
                    "question": item["text"],
                    "label": item["label"].strip().lower(),
                    "condition": condition,
                    "prediction": pred
                })
                done.add((qid, condition))

        except Exception as e:
            print(f"\nError at {split_name}, question {qid}: {e}")
            pd.DataFrame(rows).to_csv(checkpoint_file, index=False)
            raise

        # Save after each 50 questions (or near that point)
        if qid % 50 == 0:
            pd.DataFrame(rows).to_csv(checkpoint_file, index=False)

    # Save final split results
    result_df = pd.DataFrame(rows)
    result_df.to_csv(output_file, index=False)
    result_df.to_csv(checkpoint_file, index=False)

    print(f"\n{split_name} complete: {len(result_df)} predictions")
    print("Saved:", output_file)

    all_summaries.extend(calculate_metrics(output_file, split_name))

    # Save summary immediately after each completed split
    pd.DataFrame(all_summaries).to_csv(
        f"{BASE}/pope_metrics_summary.csv", index=False
    )

print("\nALL SPLITS FINISHED")
print("Total elapsed hours:", round((time.time() - start_all) / 3600, 2))
print("Metrics saved to /kaggle/working/pope_metrics_summary.csv")
print(pd.DataFrame(all_summaries).to_string(index=False))

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()
print("Model loaded successfully!")
print("Device:", model.device)

In [ ]:
import os, glob

print("Checkpoint files:")
for p in glob.glob(
    "/kaggle/input/**/pope_adversarial_checkpoint.csv",
    recursive=True
):
    print(p)

print("\nImage directory exists:",
      os.path.isdir("/kaggle/working/coco2014/val2014"))

print("Original image archive exists:",
      os.path.exists("/kaggle/working/val2014.zip"))

In [ ]:
import os

IMAGE_DIR = "/kaggle/working/coco2014/val2014"

print("Images directory exists:", os.path.isdir(IMAGE_DIR))
if os.path.isdir(IMAGE_DIR):
    print("Number of files:", len(os.listdir(IMAGE_DIR)))

In [ ]:
import pandas as pd
import os

INPUT_CSV = "/kaggle/input/datasets/umamaheshkamparaju/adversarial/pope_adversarial_checkpoint.csv"

print("Model loaded:", "model" in globals())
print("Processor loaded:", "processor" in globals())

df = pd.read_csv(INPUT_CSV)
print("Saved prediction rows:", len(df))
print("Unique questions:", df["question_id"].nunique())
print(df["condition"].value_counts())

In [ ]:
import os, json, pandas as pd
from PIL import Image, ImageFilter
from tqdm.auto import tqdm
import torch

POPE_FILE = "/kaggle/input/datasets/umamaheshkamparaju/for-pope/coco_pope_adversarial.json"
IMAGE_DIR = "/kaggle/working/coco2014/val2014"
INPUT_CSV = "/kaggle/input/datasets/umamaheshkamparaju/adversarial/pope_adversarial_checkpoint.csv"

CHECKPOINT_OUT = "/kaggle/working/pope_adversarial_checkpoint.csv"
FINAL_FILE = "/kaggle/working/pope_adversarial_full.csv"

# Load existing predictions
df = pd.read_csv(INPUT_CSV)
rows = df.to_dict("records")
done = {(int(r["question_id"]), r["condition"]) for r in rows}

print("Existing rows:", len(rows))
print("Completed question IDs:", len(set(q for q, _ in done)))

# Load questions
with open(POPE_FILE) as f:
    questions = [json.loads(line) for line in f if line.strip()]

# Reuse the same inference setup as the original experiment
def predict_one(image, question):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question + " Answer only yes or no."}
        ]
    }]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[image],
        padding=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=8, do_sample=False
        )

    generated = output[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(
        generated, skip_special_tokens=True
    )[0].strip().lower()

    if answer.startswith("yes"):
        return "yes"
    if answer.startswith("no"):
        return "no"
    return "uncertain"

# Resume missing pairs only
for item in tqdm(questions, desc="Resume adversarial"):
    qid = int(item["question_id"])
    missing = [
        c for c in ("original", "blurred")
        if (qid, c) not in done
    ]
    if not missing:
        continue

    image_path = os.path.join(IMAGE_DIR, item["image"])
    if not os.path.exists(image_path):
        print("Missing image:", item["image"])
        continue

    try:
        with Image.open(image_path) as im:
            original = im.convert("RGB")

        blurred = original.filter(ImageFilter.GaussianBlur(radius=3))

        for condition in missing:
            image = original if condition == "original" else blurred
            pred = predict_one(image, item["text"])

            rows.append({
                "question_id": qid,
                "image": item["image"],
                "question": item["text"],
                "label": item["label"].strip().lower(),
                "condition": condition,
                "prediction": pred
            })
            done.add((qid, condition))

        if qid % 25 == 0:
            pd.DataFrame(rows).to_csv(CHECKPOINT_OUT, index=False)

    except Exception:
        pd.DataFrame(rows).to_csv(CHECKPOINT_OUT, index=False)
        raise

# Save completed results
result = pd.DataFrame(rows)
result.to_csv(CHECKPOINT_OUT, index=False)
result.to_csv(FINAL_FILE, index=False)

print("\nCompleted!")
print("Total rows:", len(result))
print("Unique questions:", result["question_id"].nunique())
print(result.groupby(["condition", "prediction"]).size())
print("Final file:", FINAL_FILE)

In [ ]:
import os, json

POPE_FILE = "/kaggle/input/datasets/umamaheshkamparaju/for-pope/coco_pope_adversarial.json"
IMAGE_DIR = "/kaggle/working/coco2014/val2014"

with open(POPE_FILE) as f:
    questions = [json.loads(line) for line in f if line.strip()]

# Find questions still missing one or both predictions
import pandas as pd
checkpoint = pd.read_csv(
    "/kaggle/input/datasets/umamaheshkamparaju/adversarial/pope_adversarial_checkpoint.csv"
)

done = {(int(r.question_id), r.condition) for r in checkpoint.itertuples()}

missing_questions = [
    q for q in questions
    if any((int(q["question_id"]), c) not in done
           for c in ("original", "blurred"))
]

missing_images = sorted({
    q["image"] for q in missing_questions
    if not os.path.exists(os.path.join(IMAGE_DIR, q["image"]))
})

print("Questions still needing predictions:", len(missing_questions))
print("Distinct missing image files:", len(missing_images))
print("First 20 missing files:", missing_images[:20])

In [ ]:

import os
import glob

IMAGE_DIR = "/kaggle/working/coco2014/val2014"

sample_names = [
    "COCO_val2014_000000003845.jpg",
    "COCO_val2014_000000008749.jpg",
    "COCO_val2014_000000015738.jpg",
]

print("Image directory exists:", os.path.isdir(IMAGE_DIR))
print("COCO val2014 images:", len(
    glob.glob(os.path.join(IMAGE_DIR, "COCO_val2014_*.jpg"))
))

for name in sample_names:
    matches = (
        glob.glob("/kaggle/input/**/" + name, recursive=True)
        + glob.glob("/kaggle/working/**/" + name, recursive=True)
    )
    print(f"{name} -> {matches[:5] if matches else 'NOT FOUND'}")


In [ ]:
print("Model loaded:", "model" in globals() and model is not None)
print("Processor loaded:", "processor" in globals() and processor is not None)

In [ ]:
import os

for root in ["/kaggle/input/datasets", "/kaggle/input"]:
    print(f"\n{root}:")
    if os.path.exists(root):
        print(os.listdir(root)[:30])
    else:
        print("Not found")

In [ ]:

import os

roots = [
    "/kaggle/input/datasets/umamaheshkamparaju",
    "/kaggle/input/datasets/awsaf49",
    "/root/.cache/huggingface",
    "/kaggle/working",
]

for root in roots:
    print(f"\n--- {root} ---")
    if os.path.exists(root):
        for item in os.listdir(root)[:30]:
            path = os.path.join(root, item)
            print(item, "(folder)" if os.path.isdir(path) else "(file)")
    else:
        print("NOT FOUND")


In [ ]:

import os
import zipfile

ZIP_PATH = "/kaggle/working/val2014.zip"
EXTRACT_DIR = "/kaggle/working/coco2014"
IMAGE_DIR = f"{EXTRACT_DIR}/val2014"

if os.path.isdir(IMAGE_DIR) and len(os.listdir(IMAGE_DIR)) >= 40000:
    print("COCO images already restored.")
elif os.path.isfile(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH):
    print("Extracting existing archive...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Archive missing. Downloading COCO val2014...")
    import subprocess
    result = subprocess.run([
        "wget", "-c",
        "https://images.cocodataset.org/zips/val2014.zip",
        "-O", ZIP_PATH
    ])
    if result.returncode != 0:
        raise RuntimeError("Download failed; check Kaggle internet access.")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    print("Extraction complete.")

print("Image directory:", os.path.isdir(IMAGE_DIR))
if os.path.isdir(IMAGE_DIR):
    print("Image count:", len(os.listdir(IMAGE_DIR)))


In [ ]:
import os
import pandas as pd
import torch

IMAGE_DIR = "/kaggle/working/coco2014/val2014"
CHECKPOINT = "/kaggle/input/datasets/umamaheshkamparaju/adversarial/pope_adversarial_checkpoint.csv"

print("Model loaded:", "model" in globals() and model is not None)
print("Processor loaded:", "processor" in globals() and processor is not None)
print("GPU available:", torch.cuda.is_available())
print("COCO images:", len(os.listdir(IMAGE_DIR)))

df = pd.read_csv(CHECKPOINT)
print("Checkpoint rows:", len(df))
print("Completed questions:", df["question_id"].nunique())

In [ ]:

%pip install -q -U accelerate
%pip install -q git+https://github.com/huggingface/transformers

import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

print("Model loaded:", model is not None)
print("Processor loaded:", processor is not None)
print("Device:", model.device)


In [ ]:
import os
import pandas as pd
import torch

IMAGE_DIR = "/kaggle/working/coco2014/val2014"
CHECKPOINT = "/kaggle/input/datasets/umamaheshkamparaju/adversarial/pope_adversarial_checkpoint.csv"

print("Model loaded:", "model" in globals() and model is not None)
print("Processor loaded:", "processor" in globals() and processor is not None)
print("GPU available:", torch.cuda.is_available())
print("COCO images:", len(os.listdir(IMAGE_DIR)))

df = pd.read_csv(CHECKPOINT)
print("Checkpoint rows:", len(df))
print("Completed questions:", df["question_id"].nunique())

In [ ]:

import os, json, torch
import pandas as pd
from PIL import Image, ImageFilter
from tqdm.auto import tqdm

POPE_FILE = "/kaggle/input/datasets/umamaheshkamparaju/for-pope/coco_pope_adversarial.json"
IMAGE_DIR = "/kaggle/working/coco2014/val2014"
INPUT_CSV = "/kaggle/input/datasets/umamaheshkamparaju/adversarial/pope_adversarial_checkpoint.csv"
CHECKPOINT_OUT = "/kaggle/working/pope_adversarial_checkpoint_resumed.csv"
FINAL_FILE = "/kaggle/working/pope_adversarial_full.csv"

assert model is not None and processor is not None
assert os.path.isdir(IMAGE_DIR)

with open(POPE_FILE, encoding="utf-8") as f:
    questions = [json.loads(line) for line in f if line.strip()]

df = pd.read_csv(INPUT_CSV)
rows = df.to_dict("records")
done = {
    (int(r["question_id"]), str(r["condition"]))
    for r in rows
    if str(r.get("prediction", "")).lower() in ("yes", "no", "uncertain")
}

print("Existing rows:", len(rows))
print("Completed pairs:", len(done))
print("Questions in split:", len(questions))

def predict_one(image, question):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question + " Answer only yes or no."}
        ]
    }]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[image],
        padding=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=8, do_sample=False
        )

    generated = output[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(
        generated, skip_special_tokens=True
    )[0].strip()

    a = answer.lower().strip(" .,!:")
    if a.startswith("yes"):
        pred = "yes"
    elif a.startswith("no"):
        pred = "no"
    else:
        pred = "uncertain"
    return answer, pred

missing_images = set()
new_count = 0

for item in tqdm(questions, desc="Resume adversarial"):
    qid = int(item["question_id"])
    missing_conditions = [
        c for c in ("original", "blurred") if (qid, c) not in done
    ]
    if not missing_conditions:
        continue

    image_path = os.path.join(IMAGE_DIR, item["image"])
    if not os.path.isfile(image_path):
        missing_images.add(item["image"])
        continue

    with Image.open(image_path) as im:
        original = im.convert("RGB")

    blurred = original.filter(ImageFilter.GaussianBlur(radius=3))

    for condition in missing_conditions:
        image = original if condition == "original" else blurred
        raw, pred = predict_one(image, item["text"])

        rows.append({
            "question_id": qid,
            "image": item["image"],
            "question": item["text"],
            "label": str(item["label"]).strip().lower(),
            "condition": condition,
            "raw_answer": raw,
            "prediction": pred,
            "error": ""
        })
        done.add((qid, condition))
        new_count += 1

    if new_count and new_count % 50 == 0:
        pd.DataFrame(rows).to_csv(CHECKPOINT_OUT, index=False)

    # Save progress after each question
    pd.DataFrame(rows).to_csv(CHECKPOINT_OUT, index=False)

result = pd.DataFrame(rows)
result.to_csv(FINAL_FILE, index=False)

expected = {
    (int(q["question_id"]), c)
    for q in questions for c in ("original", "blurred")
}
actual = {
    (int(r["question_id"]), str(r["condition"]))
    for r in rows
}

print("\nNew predictions:", new_count)
print("Total CSV rows:", len(result))
print("Unique questions:", result["question_id"].nunique())
print("Missing image files:", len(missing_images))
print("Missing prediction pairs:", len(expected - actual))
print("Saved:", FINAL_FILE)

if missing_images:
    print("First missing images:", sorted(missing_images)[:20])

if expected - actual:
    print("WARNING: adversarial split is not complete yet.")
else:
    print("SUCCESS: all adversarial question-condition pairs are present.")


In [ ]:
import torch

print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("Model device:", next(model.parameters()).device)

In [ ]:
import torch, time

print("GPUs:", torch.cuda.device_count())
print("Model:", next(model.parameters()).device)
print("Model loaded:", model is not None)

In [ ]:
import pandas as pd
import os

paths = [
    "/kaggle/input/datasets/umamaheshkamparaju/2nd-part/pope_random_full.csv",
    "/kaggle/input/datasets/umamaheshkamparaju/2nd-part/pope_popular_full.csv",
    "/kaggle/input/datasets/umamaheshkamparaju/adversarial/pope_adversarial_checkpoint.csv",
]

for p in paths:
    print(os.path.exists(p), p)

In [ ]:
import pandas as pd
import os

files = {
    "Random": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_random_full.csv",
    "Popular": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_popular_full.csv",
    "Adversarial": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_adversarial_full (1).csv"
}

for name, path in files.items():
    print("\n" + "=" * 50)
    print(name, "| File exists:", os.path.exists(path))

    if os.path.exists(path):
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        display(df.head(2))

In [ ]:
import os, time, re
import pandas as pd
from PIL import Image
from io import BytesIO
import torch

files = {
    "random": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_random_full.csv",
    "popular": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_popular_full.csv",
    "adversarial": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_adversarial_full (1).csv"
}

# COCO images from your previous setup
IMG_DIR = "/kaggle/working/coco2014/val2014"
QUALITY = 30
N_QUESTIONS = 100

# Verify setup before inference
assert torch.cuda.is_available(), "GPU unavailable"
assert next(model.parameters()).device.type == "cuda", "Model is not on GPU"
assert os.path.isdir(IMG_DIR), f"Image directory not found: {IMG_DIR}"

def jpeg_compress(image, quality=30):
    buffer = BytesIO()
    image.convert("RGB").save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")

def predict_jpeg(image, question):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question + " Answer only yes or no."}
        ]
    }]

    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[image],
        padding=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=8, do_sample=False
        )

    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(
        new_tokens, skip_special_tokens=True
    )[0].strip().lower()

    # Parse answer conservatively
    match = re.search(r"\b(yes|no)\b", answer)
    return match.group(1) if match else "uncertain"

all_results = []

for split, path in files.items():
    df = pd.read_csv(path)

    # One row per question, using its original condition as the source
    base = (
        df[df["condition"] == "original"]
        .drop_duplicates("question_id")
        .head(N_QUESTIONS)
    )

    results = []
    start = time.time()

    for i, row in enumerate(base.itertuples(index=False), 1):
        image_path = os.path.join(IMG_DIR, row.image)

        if not os.path.isfile(image_path):
            raise FileNotFoundError(
                f"Image missing: {image_path}. Stop; check IMG_DIR."
            )

        with Image.open(image_path) as im:
            compressed = jpeg_compress(im, QUALITY)

        pred = predict_jpeg(compressed, row.question)

        results.append({
            "question_id": row.question_id,
            "image": row.image,
            "question": row.question,
            "label": str(row.label).lower(),
            "condition": f"jpeg_q{QUALITY}",
            "prediction": pred
        })

        if i % 10 == 0:
            elapsed = time.time() - start
            print(f"{split}: {i}/{len(base)} | {elapsed:.1f}s elapsed")

    out = f"/kaggle/working/pope_{split}_jpeg_q{QUALITY}_pilot.csv"
    pd.DataFrame(results).to_csv(out, index=False)
    all_results.extend(results)

    print(f"\n{split} complete: {len(results)} questions")
    print(f"Time: {(time.time()-start)/60:.1f} min")
    print("Saved:", out)

print("\nPILOT COMPLETE — total predictions:", len(all_results))

In [ ]:
import os

root = "/kaggle/input"

for current, dirs, files in os.walk(root):
    if "COCO_val2014_000000310196.jpg" in files:
        print("FOUND:", os.path.join(
            current, "COCO_val2014_000000310196.jpg"
        ))
        break
else:
    print("Sample image not found")

In [ ]:
import os
from PIL import Image

IMG_DIR = "/kaggle/input/datasets/jeffaudi/coco-2014-dataset-for-yolov3/coco2014/images/val2014"

test_image = os.path.join(IMG_DIR, "COCO_val2014_000000310196.jpg")

print("Image exists:", os.path.isfile(test_image))

with Image.open(test_image) as im:
    print("Image size:", im.size)
    display(im.copy())

In [ ]:
import os, re, time
import pandas as pd
import torch
from PIL import Image
from io import BytesIO

IMG_DIR = "/kaggle/input/datasets/jeffaudi/coco-2014-dataset-for-yolov3/coco2014/images/val2014"

FILES = {
    "random": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_random_full.csv",
    "popular": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_popular_full.csv",
    "adversarial": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_adversarial_full (1).csv"
}

QUALITY = 30
N = 100

assert os.path.isdir(IMG_DIR), "Image directory missing"
assert next(model.parameters()).device.type == "cuda", "Model not on GPU"

def compress_jpeg(img, quality=30):
    buf = BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    with Image.open(buf) as im:
        return im.convert("RGB").copy()

def predict_image(img, question):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": question + " Answer only yes or no."}
        ]
    }]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[img],
        padding=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=8, do_sample=False)

    tokens = output[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(tokens, skip_special_tokens=True)[0].lower()
    match = re.search(r"\b(yes|no)\b", answer)
    return match.group(1) if match else "uncertain"

for split, path in FILES.items():
    print(f"\nStarting {split}...")
    df = pd.read_csv(path)

    base = (
        df[df["condition"].astype(str).str.lower() == "original"]
        .drop_duplicates("question_id")
        .head(N)
    )
    assert len(base) == N, f"{split}: expected {N}, got {len(base)}"

    results = []
    start = time.time()

    for i, row in enumerate(base.itertuples(index=False), 1):
        img_path = os.path.join(IMG_DIR, row.image)
        if not os.path.isfile(img_path):
            raise FileNotFoundError(img_path)

        with Image.open(img_path) as im:
            img = compress_jpeg(im, QUALITY)

        pred = predict_image(img, row.question)
        results.append({
            "question_id": row.question_id,
            "image": row.image,
            "question": row.question,
            "label": str(row.label).strip().lower(),
            "condition": f"jpeg_q{QUALITY}",
            "prediction": pred
        })

        if i % 10 == 0:
            print(f"{split}: {i}/{N}, {time.time()-start:.1f}s")

    out = f"/kaggle/working/pope_{split}_jpeg_q{QUALITY}_pilot.csv"
    pd.DataFrame(results).to_csv(out, index=False)
    print(f"Saved: {out} | Time: {(time.time()-start)/60:.2f} min")

print("\nPilot complete!")

In [ ]:
print("model exists:", "model" in globals())
print("processor exists:", "processor" in globals())

In [ ]:
import os, re, time
import pandas as pd
import torch
from PIL import Image
from io import BytesIO

IMG_DIR = "/kaggle/input/datasets/jeffaudi/coco-2014-dataset-for-yolov3/coco2014/images/val2014"

FILES = {
    "random": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_random_full.csv",
    "popular": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_popular_full.csv",
    "adversarial": "/kaggle/input/datasets/umamaheshkamparaju/compression/pope_adversarial_full (1).csv"
}

QUALITY = 30
OUT_DIR = "/kaggle/working/jpeg_q30"
os.makedirs(OUT_DIR, exist_ok=True)

assert torch.cuda.is_available()
assert "model" in globals() and "processor" in globals()
assert next(model.parameters()).device.type == "cuda"

def jpeg_compress(img, quality=30):
    buf = BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    with Image.open(buf) as im:
        return im.convert("RGB").copy()

def predict_image(img, question):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": question + " Answer only yes or no."}
        ]
    }]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[img],
        padding=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False)

    tokens = out[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(tokens, skip_special_tokens=True)[0].lower()
    match = re.search(r"\b(yes|no)\b", answer)
    return match.group(1) if match else "uncertain"

for split, path in FILES.items():
    print(f"\n===== {split.upper()} =====", flush=True)
    df = pd.read_csv(path)

    base = (
        df[df["condition"].astype(str).str.lower() == "original"]
        .drop_duplicates("question_id")
        .reset_index(drop=True)
    )

    out_path = os.path.join(OUT_DIR, f"pope_{split}_jpeg_q30.csv")

    # Resume previously saved predictions, including pilot rows
    if os.path.exists(out_path):
        saved = pd.read_csv(out_path)
        done = set(saved["question_id"].astype(str))
        results = saved.to_dict("records")
        print(f"Resuming: {len(done)} already saved", flush=True)
    else:
        done, results = set(), []

    start = time.time()
    for i, row in enumerate(base.itertuples(index=False), 1):
        qid = str(row.question_id)
        if qid in done:
            continue

        img_path = os.path.join(IMG_DIR, row.image)
        if not os.path.isfile(img_path):
            raise FileNotFoundError(img_path)

        with Image.open(img_path) as im:
            img = jpeg_compress(im, QUALITY)

        pred = predict_image(img, row.question)
        results.append({
            "question_id": row.question_id,
            "image": row.image,
            "question": row.question,
            "label": str(row.label).strip().lower(),
            "condition": "jpeg_q30",
            "prediction": pred
        })
        done.add(qid)

        # Save every 25 predictions to protect progress
        if len(results) % 25 == 0:
            pd.DataFrame(results).to_csv(out_path, index=False)

        if len(done) % 50 == 0:
            elapsed = time.time() - start
            print(
                f"{split}: {len(done)}/{len(base)} | "
                f"{elapsed/60:.1f} min this session",
                flush=True
            )

    pd.DataFrame(results).to_csv(out_path, index=False)
    print(f"Saved {len(results)} rows -> {out_path}", flush=True)

print("\nALL SPLITS FINISHED", flush=True)